# One describer: the host's Rust describer as wasm, beside the TS port

Ruling 66's spike; the argument and the five answers are `docs/plan/repl/ONE_DESCRIBER_WASM_SPIKE.md`, which cites
this notebook for its figures. A decision (Shane's) rests on them, so they are kept here.

window.ml describes an `ml.sh` call in TypeScript (`describeCall`, `src/shell/program-manifest.ts`); window-ml-host
describes it again in Rust (`describe_call`, `src/manifest.rs`), and the shared vectors hold the two together. The
question: could the extension run the Rust describer compiled to `wasm32-unknown-unknown` and drop its own port?

**Method.** `one-describer/wrapper/` pulls the host's `src/manifest.rs` and `src/script_grammar.rs` in unmodified by
`#[path]` behind a raw ABI (JSON in, JSON out; no wasm-bindgen; no wasm-opt). `build.sh` builds it at opt `z`, `s`
and `3`. Then, from `notebooks/shell/one-describer/` with `node --import tsx`:

- `vectors.mjs <opt>`: every shared vector row through both describers, and each against the recorded answer.
- `fuzz.mjs 2000 <seed> <opt>`: 2,000 calls per program mutated from the vectors (swap, duplicate and drop
  arguments, flags spliced from the manifest, `--`, `-`, `=`, backslash-newline, CR-LF, astral and combining
  characters, NUL, lone surrogates, long strings), both describers, every disagreement grouped by the fields that
  differ and minimised.
- `sizes.mjs`; `bench.mjs` (compile, instantiate, per-call time); `bench2.mjs` (per program, and with the manifest
  JSON pre-serialised).

Normalisation before comparing: TS effects go through `effectKey`, as the vector test does; effect lists are sorted
(Rust sorts bytewise, JS by UTF-16 unit); object keys are sorted.

In [1]:
import json
from pathlib import Path
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "shell"
OUT = HERE / "one-describer" / "out"
DATA = HERE / "data" / "one-describer"
VECTOR_FIELDS = {"described", "effects", "pathArgs", "actsOnDir", "runner", "okCodes", "echoes", "recursive", "hardened"}
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 120)

def read_out():
    rows = [{"kind": "build", **json.loads((OUT / "build.json").read_text())},
            {"kind": "sizes", **json.loads((OUT / "sizes.json").read_text())},
            {"kind": "bench", **json.loads((OUT / "bench.json").read_text())},
            {"kind": "bench-by-program", **json.loads((OUT / "bench-by-program.json").read_text())}]
    for p in sorted(OUT.glob("vectors-*.json")):
        v = json.loads(p.read_text())
        rows.append({"kind": "vectors", **v})
    for p in sorted(OUT.glob("fuzz-*.json")):
        f = json.loads(p.read_text())
        # Keep each group's count, programs and up to 3 examples: the full examples stay in out/.
        f["groups"] = [{**g, "examples": g.get("examples", [])[:3]} for g in f["groups"]]
        rows.append({"kind": "fuzz", **f})
    return rows

rows = wmlnb.extract(DATA / "results.json", read_out, source={"harness": "notebooks/shell/one-describer/", "out": "notebooks/shell/one-describer/out/*.json"})
of = lambda k: [r for r in rows if r["kind"] == k]
build = of("build")[0]
pd.DataFrame(wmlnb.provenance(DATA / "results.json")).assign(host=build["host"][:12], rustc=build["rustc"])

,extract,sha256,rows,extracted_at,commit,harness,out,host,rustc
0,notebooks/shell/data/one-describer/results.json,6025a10005322a5e,17,2026-11-20T08:02:48+00:00,a771c714ceb4,notebooks/shell/one-describer/,notebooks/shell/one-describer/out/*.json,cd82e8cc0cf1,rustc 1.99.1 (58cf8f9c3 2026-10-13)


## 1. The vectors: 0 differences is the bar

Every row of `tests/fixtures/program-calls/` and `program-calls-flavoured/`, through both describers, at every opt
level. `wasmVsTs` counts rows where any output field differs; the two `…VsRecorded` columns compare the nine fields the
vector tests compare.

In [2]:
pd.DataFrame([{k: v[k] for k in ("opt", "rows", "plain", "flavoured", "wasmVsTsDiffs", "wasmVsRecorded", "tsVsRecorded")} for v in of("vectors")])

,opt,rows,plain,flavoured,wasmVsTsDiffs,wasmVsRecorded,tsVsRecorded
0,3,532,507,25,0,0,0
1,s,532,507,25,0,0,0
2,z,532,507,25,0,0,0


## 2. The differential fuzz: does the second port catch what the vectors do not?

A disagreement is **verdict-level** when it touches a field the vectors compare (described, effects, path arguments,
the hardened argv, …). Anything else is the `why` text or an error from the wasm side.

In [3]:
def level(cause):
    if cause.startswith("WASM-ERROR"): return "wasm error"
    fields = set(cause.split(" / ")[0].split("+"))  # fuzz.mjs joins the differing fields with "+"
    return "verdict" if fields & VECTOR_FIELDS else "text"

# A split on the wrong separator would class every cause as text and make the verdict count 0 by construction.
assert level("described+effects") == "verdict" and level("why+effects / x") == "verdict"
assert level("why / why:non-ascii-quoting") == "text" and level("WASM-ERROR / lone-surrogate-in-argv") == "wasm error"

runs = []
for f in of("fuzz"):
    by = {"verdict": 0, "text": 0, "wasm error": 0}
    for g in f["groups"]: by[level(g["cause"])] += g["count"]
    runs.append({"opt": f["opt"], "seed": f["seed"], "calls": f["total"], "described by TS": f["tsDescribed"], "disagreements": f["disagreements"], **by})
runs = pd.DataFrame(runs).sort_values(["opt", "seed"])
display(runs)
print(f"{len(runs)} runs, {runs.calls.sum():,} calls, verdict-level disagreements: {runs.verdict.sum()}")

,opt,seed,calls,described by TS,disagreements,verdict,text,wasm error
0,3,1,34000,12277,159,0,155,4
1,3,2,34000,12234,148,0,141,7
2,3,3,34000,12347,152,0,145,7
3,s,1,34000,12277,159,0,155,4
4,s,2,34000,12234,148,0,141,7
5,s,3,34000,12347,152,0,145,7
6,z,1,34000,12277,159,0,155,4
7,z,2,34000,12234,148,0,141,7
8,z,3,34000,12347,152,0,145,7
9,z,7,34000,12128,153,0,146,7


10 runs, 340,000 calls, verdict-level disagreements: 0


In [4]:
causes = pd.DataFrame([{"opt": f["opt"], "seed": f["seed"], "cause": g["cause"], "level": level(g["cause"]), "count": g["count"]} for f in of("fuzz") for g in f["groups"]])
causes.pivot_table(index=["level", "cause"], columns=["opt", "seed"], values="count", fill_value=0)

opt                                                   3                 s                 z                  
seed                                                  1     2     3     1     2     3     1     2     3     7
level      cause                                                                                             
text       why / why:ascii-only                     6.0  10.0  10.0   6.0  10.0  10.0   6.0  10.0  10.0  10.0
           why / why:astral-char-split-in-cluster  80.0  68.0  74.0  80.0  68.0  74.0  80.0  68.0  74.0  79.0
           why / why:non-ascii-quoting             69.0  63.0  61.0  69.0  63.0  61.0  69.0  63.0  61.0  57.0
wasm error WASM-ERROR / lone-surrogate-in-argv      4.0   7.0   7.0   4.0   7.0   7.0   4.0   7.0   7.0   7.0

One example of each cause (seed 1, opt `z`). The astral case is TS splitting a surrogate pair when it quotes one
character of a flag cluster; the quoting case is Rust's `{:?}` (`\\0`, `\\u{301}`) against JSON (`\\u0000`, the raw
mark); a lone surrogate is refused by serde_json, so the wasm side errors where TS describes.

In [5]:
import re
# A lone surrogate is one of the findings, and Jupyter cannot send one as UTF-8: show each value as JSON with surrogates escaped.
shown = lambda x: re.sub("[\ud800-\udfff]", lambda m: f"\\u{ord(m.group()):04x}", json.dumps(x, ensure_ascii=False))
f1 = next(f for f in of("fuzz") if f["opt"] == "z" and f["seed"] == 1)
pd.DataFrame([{"cause": g["cause"], **{k: shown(g["examples"][0].get(k)) for k in ("argv", "wasm", "ts")}} for g in f1["groups"] if g["examples"]])

,cause,argv,wasm,ts
0,why / why:astral-char-split-in-cluster,"[""cat"", ""-vet🙂""]","{""why"": [""flag -🙂 (in -vet🙂) is not in the manifest""]}","{""why"": [""flag -\ud83d (in -vet🙂) is not in the manifest""]}"
1,why / why:non-ascii-quoting,"[""find"", ""-printf́""]","{""why"": [""find expression \""-printf\\u{301}\"" is not in the manifest""]}","{""why"": [""find expression \""-printf́\"" is not in the manifest""]}"
2,why / why:ascii-only,"[""gh"", ""\u0000""]","{""why"": [""the program's version was not reported"", ""gh has no subcommand \""\\0\"" in its manifest""]}","{""why"": [""the program's version was not reported"", ""gh has no subcommand \""\\u0000\"" in its manifest""]}"
3,WASM-ERROR / lone-surrogate-in-argv,"[""cat"", ""\ud83d""]","""request: unexpected end of hex escape at line 1 column 23""",null


## 3. Size

The TS describer is `describeCall` and what it imports, bundled and minified by esbuild. The extension already ships
Pyodide's `pyodide.asm.wasm` at about 9.6 MB.

In [6]:
s = of("sizes")[0]
sizes = pd.DataFrame([{"build": k, **v} for k, v in s.items() if k != "kind"]).set_index("build")
ts = sizes.loc["ts-describeCall"]
sizes.assign(**{f"x TS {c}": (sizes[c] / ts[c]).round(1) for c in ("raw", "gzip", "brotli")})

,brotli,gzip,raw,x TS raw,x TS gzip,x TS brotli
build,,,,,,
ts-describeCall,5172,5625,14911,1.0,1.0,1.0
wasm-3,92348,114784,335351,22.5,20.4,17.9
wasm-s,80998,95110,244675,16.4,16.9,15.7
wasm-z,76661,88764,219135,14.7,15.8,14.8


## 4. Time (Node, this machine)

Compile and instantiate: median of 20 fresh compiles. Per call: every vector row, 25 timed passes after 5 warm-up
passes. None of this is Chrome's service worker, which is unmeasured.

In [7]:
b = of("bench")[0]
print(b["node"], b["platform"], "load average", [round(x, 2) for x in b["loadavg"]])
display(pd.DataFrame(b["instantiate"]).T.round(2))
pd.DataFrame(b["latency"]).T.round(1)

v26.12.1 darwin arm64 load average [2.09, 5.73, 7.33]


,compileMax,compileMin,compileMs,instantiateMs,totalMs
3,4.26,0.23,0.30,0.04,0.33
s,0.21,0.16,0.18,0.03,0.20
z,1.48,0.16,0.17,0.03,0.20


,medianUs,n,p95Us
ts,1.7,13300.0,3.3
ts-describeCall-only,1.6,13300.0,3.0
wasm-3,31.1,13300.0,302.0
wasm-s,32.9,13300.0,335.8
wasm-z,39.5,13300.0,402.5


Where a wasm call's time goes (opt `s`): it grows with the manifest, because the whole manifest crosses as JSON and
serde parses it on every call. Pre-serialising it in JS saves only the stringify. A handle ABI (parse once, describe
by handle) was not built, so the describer's own cost is not separated.

In [8]:
bp = of("bench-by-program")[0]
per = pd.DataFrame([{"program": p, "manifest bytes": v["manifestJsonBytes"], "ts µs": v["ts"]["medianUs"],
                     "wasm µs": v["stringifyEachCall"]["medianUs"], "wasm pre-serialised µs": v["preSerialised"]["medianUs"]}
                    for p, v in bp["byProgram"].items()]).sort_values("manifest bytes")
display(per.set_index("program"))
print("all rows, median / p95 µs:", bp["all"])

,manifest bytes,ts µs,wasm µs,wasm pre-serialised µs
program,,,,
rmdir,262,1.2,5.9,5.8
unlink,273,1.1,5.9,5.5
cat,283,1.2,7.6,6.5
wc,284,1.2,6.4,6.5
mv,300,1.4,7.0,7.0
rm,331,1.3,6.8,6.6
head,355,1.3,7.0,6.8
awk,372,1.6,13.2,8.3
cp,379,1.9,8.6,7.6


all rows, median / p95 µs: {'each': {'med': 49.2, 'p95': 489.8}, 'pre': {'med': 38.8, 'p95': 369.7}}


## 5. What the cross-check caught in review

The number ruling 66's answer turns on: of the describer's PRs in both repos, how often did the SECOND port catch a
defect one copy would not have. Read from GitHub only (reviews, review comments, issue comments, commits); each
finding classed A (caught by comparing the ports), B (one copy would have caught it: a vector, the red team, a review
of one side), C (a cost of having two: one port lagging, an order mismatch, a pin bump, no defect), or unclear.

In [9]:
def read_history():
    return json.loads((OUT / "history.json").read_text())

hist = wmlnb.extract(DATA / "history.json", read_history, source={"method": "gh only: PRs touching the describer's files in parawanderer/window-ml and parawanderer/window-ml-host; see the spike doc"})
h = pd.DataFrame(hist)
display(h["class"].value_counts().rename("findings").to_frame())
h[h["class"] == "A"]

,findings
class,
B,15
C,14
A,1
unclear,1


,class,pr,quote,reasoning,url
0,A,window-ml#995,"commit 7157b94d: ""validateManifest refuses an unknown version key, as the host's VersionSpec does ... The version ob...",TS validator was wrong (accepted unknown keys in `version`); found by comparing with the host reader's deny_unknown_...,https://github.com/parawanderer/window-ml/pull/995
